In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
products_bronze_path = f"{BRONZE_PATH}/products"
products_silver_path = f"{SILVER_PATH}/products"

In [0]:
df_products_bronze = spark.read.format("delta") \
    .load(products_bronze_path)

In [0]:
df_products_bronze.printSchema()

In [0]:
display(df_products_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("product_id").orderBy(F.col("updated_at").desc())

df_products_silver = df_products_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
display(df_products_silver)

In [0]:
%skip
df_products_silver.write.format("delta") \
    .mode("append") \
    .save(products_silver_path)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(
    spark,
    products_silver_path
)

silver_table.alias("target") \
.merge(
    df_products_silver.alias("source"),
    "target.product_id = source.product_id"
) \
.whenMatchedUpdate(
    set = {
        "category_id": "source.category_id",
        "supplier_id": "source.supplier_id",
        "product_name": "source.product_name",
        "sku": "source.sku",
        "unit_price": "source.unit_price",
        "active": "source.active",
        "updated_at": "source.updated_at"
    }
) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
display(
    spark.read.format("delta") \
        .load(products_silver_path)
)